In [1]:
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
import random
from tqdm.auto import tqdm
import pandas as pd


In [2]:
val_data = []
with open("dev.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    val_data.append(json.loads(line))


test_data = []
with open("test.jsonl", "r") as f:
    data = f.readlines()

for line in data:
    test_data.append(json.loads(line))

In [3]:

BASE_MODEL_ID = "microsoft/phi-2"
# torch.set_default_device(device)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
print(  val_data[0].keys(), )
print(val_data[0]['question']['choices'], val_data[0]['question']['stem'])
# print(  test_data[0]['question']['choices'][3], )


# print(test_data[0].keys())


dict_keys(['id', 'question', 'answerKey', 'fact1', 'fact2', 'combinedfact', 'formatted_question'])
[{'text': 'sand', 'label': 'A', 'para': 'Generally if there is a beach on the shore, it is beautiful sand. What sand there is, is liberally peppered with seaweed. Faith is to the human what sand is to the ostrich. Sun, Sand and a perfect climate contribute to a lively youthful atmosphere. Simply described, a sand tray is a small sand box designed for indoor use. Blast Describes a shot from a sand bunker. Also the term sand is used interchangeably. Hookworms are often found in the soil or sand in moderate climates. What sand remains is but residue. Generally such soils are sands or loamy sands.'}, {'text': 'occurs over a wide range', 'label': 'B', 'para': "Engineering is a wide range of activities that can be described best in terms of functions. Climate A wide range of climatic conditions are present in the large geographical range of redbud. Aroids grow all over the world and occur in a 

In [5]:
from string import Template
prompt_template= Template('''Answer the following question using the context below by selecting the most likely option (A, B, C, D, E, F, G or H):\n
$question
context: $context      
                                                        
$options
$question
''')


# prompt_template2= Template('''Instruct: Answer the following question using the context provided, reason over it because only one of the context is relevant . Please generate only answer choice (1, 2, 3, 4, 5, 6, 7 or 8) without any explanations\n
# $question
# context: $context      
                                                        
# $options
# $question
# ''')


In [6]:

# data[0]
submission = {"answers":[]}
option_header = ["option A ", "option B ", "option C ", "option D ", "option E ","option F ", "option G ", "option H " ]
map_ans = {"A":1, "B":2, "C":3, "D":4, "E":5, "F":6, "G":7, "H":8}
pbar = tqdm(range(len(val_data)))
for example in val_data:
    options = []
    opts = []
    context = ''
    for i, sample in enumerate(example['question']['choices']):
        # print(key)
        
    
        options.append(option_header[i]+ sample['text'])
        opts.append((sample['text'], option_header[i].split("option")[1]))
        context += '\n'+ sample['para']

    # combined_fact = example['combinedfact']

    combined_fact = example["fact1"] + '\n' + example["fact2"]


    prompt_sample = prompt_template.substitute( question  = example['question']['stem'], context=  combined_fact, options = "\n".join(options))

    # print(prompt_sample)

    input  = tokenizer(prompt_sample, return_tensors="pt").to("cuda")
    # print(input)
    out = model.generate(**input,  max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.batch_decode(out)[0]
    answer_only = text[len(prompt_sample):]
    # answer_only = text


    try:
        id  = answer_only.split(" correct option is ")[1][0]
        id = int(map_ans[id])
        


    except:

        id  = random.randint(1,8)

    submission["answers"].append(id)
    pd.DataFrame(submission).to_csv("phik1_qasc_f1f2.csv")
    
    pbar.set_description(f"Pred: {id:.4f}")
    pbar.update(1)
pbar.close()
# print(answer_only)

  0%|          | 0/926 [00:00<?, ?it/s]

In [7]:
answer_only.split(" correct option is ")[1][0]

'D'

In [8]:
# print(prompt_sample +"\nOption ")
test = prompt_sample +"\nCorrect Option "
tokenizer.pad_token_id = tokenizer.eos_token_id
input  = tokenizer(test, return_tensors="pt").to("cuda")
out = model.generate(**input,  max_new_tokens=50)
text = tokenizer.batch_decode(out)[0]
print(answer_only)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Solution 0:

The correct option is D dialysis.

Renal failure means kidney failure. Kidney failure is a condition in which the kidneys are unable to filter waste products and excess fluid from the blood. Dialysis is a


In [9]:
submission["answers"]


[6,
 7,
 4,
 6,
 2,
 4,
 6,
 5,
 2,
 6,
 1,
 7,
 8,
 3,
 7,
 3,
 4,
 2,
 2,
 8,
 7,
 1,
 5,
 3,
 3,
 3,
 5,
 3,
 5,
 5,
 6,
 3,
 5,
 1,
 1,
 8,
 6,
 8,
 5,
 2,
 4,
 2,
 2,
 4,
 6,
 8,
 8,
 6,
 5,
 3,
 7,
 7,
 2,
 5,
 8,
 1,
 4,
 6,
 7,
 3,
 8,
 3,
 2,
 4,
 3,
 1,
 6,
 3,
 7,
 3,
 7,
 3,
 8,
 5,
 3,
 5,
 6,
 3,
 3,
 6,
 1,
 5,
 1,
 6,
 2,
 5,
 7,
 8,
 7,
 2,
 7,
 2,
 4,
 6,
 5,
 4,
 8,
 1,
 4,
 4,
 1,
 8,
 1,
 7,
 4,
 8,
 5,
 3,
 1,
 1,
 5,
 8,
 3,
 2,
 3,
 2,
 6,
 1,
 7,
 4,
 4,
 3,
 1,
 8,
 5,
 3,
 1,
 4,
 7,
 3,
 3,
 4,
 7,
 2,
 2,
 1,
 7,
 4,
 8,
 7,
 5,
 5,
 7,
 2,
 5,
 3,
 1,
 3,
 6,
 8,
 8,
 3,
 4,
 1,
 4,
 3,
 8,
 1,
 7,
 1,
 4,
 5,
 5,
 5,
 3,
 2,
 4,
 1,
 7,
 7,
 1,
 6,
 8,
 7,
 1,
 4,
 6,
 6,
 5,
 3,
 2,
 2,
 6,
 4,
 3,
 4,
 4,
 2,
 5,
 3,
 3,
 2,
 6,
 3,
 6,
 8,
 8,
 6,
 5,
 5,
 1,
 1,
 1,
 4,
 3,
 3,
 7,
 7,
 5,
 3,
 8,
 8,
 2,
 8,
 6,
 2,
 6,
 2,
 3,
 1,
 3,
 7,
 3,
 5,
 5,
 1,
 8,
 5,
 5,
 5,
 3,
 5,
 7,
 8,
 3,
 4,
 2,
 4,
 8,
 3,
 5,
 8,
 7,
 7,
 3,
 5,
 3,
 2,
 3,
 8,
